In [1]:
%load_ext sql
%config SqlMagic.displaycon = False

import duckdb

conn = duckdb.connect(database=":memory:")
%sql conn

%sql --file sample_data.sql

3 rows affected.

18 rows affected.

214 rows affected.

36 rows affected.

Count
36


## Data quality

The summary below describes the main data quality issues, their impact, and proposed handling. The SQL queries that follow check for these and other issues that could distort the results.

- **Duplicate activity rows:** `D002` has the same activity row twice.

  Impact: exact duplicates inflate viewing minutes and can affect brand rankings.

  Handling: remove exact duplicates, keeping one copy of each distinct row.

- **Devices without a partner:** `D017` has no partner.

  Impact: their revenue cannot be assigned to a partner, so partner payouts cannot be determined. Excluding active devices without a partner would undercount MAU and distort ARPU.
  
  Handling: keep active devices without a partner in MAU and ARPU calculations. Report their revenue separately as unassigned to a partner.

- **Activity before activation:** `D006` has activity before its activation.

  Impact: it can distort MAU, month-over-month growth, retention, activity streaks, viewing minutes, and revenue allocation.

  Handling: assume the activation date is correct and exclude earlier activity from analysis.

In [2]:
%%sql
-- partners: check partner_id for NULLs and duplicate keys.
SELECT *,
       COUNT(*) OVER (PARTITION BY partner_id) AS key_row_count
FROM partners
QUALIFY partner_id IS NULL OR key_row_count > 1
ORDER BY partner_id;


partner_id,partner_name,revenue_share_pct,key_row_count


In [3]:
%%sql
-- devices: check device_id for NULLs and duplicate keys.
SELECT *,
       COUNT(*) OVER (PARTITION BY device_id) AS key_row_count
FROM devices
QUALIFY device_id IS NULL OR key_row_count > 1
ORDER BY device_id;


device_id,brand,country,partner_id,activation_date,key_row_count


In [4]:
%%sql
-- device_activity: find rows with duplicate device_id and activity_date keys.
SELECT *
FROM device_activity
QUALIFY COUNT(*) OVER (PARTITION BY device_id, activity_date) > 1
ORDER BY device_id, activity_date;


device_id,activity_date,sessions,minutes_watched
D002,2026-09-16,4,110
D002,2026-09-16,4,110


In [5]:
%%sql
-- revenue_by_market: find rows with duplicate revenue_month, country, and product keys.
SELECT *
FROM revenue_by_market
QUALIFY COUNT(*) OVER (PARTITION BY revenue_month, country, product) > 1
ORDER BY revenue_month, country, product;


revenue_month,country,product,revenue_eur


In [6]:
%%sql
-- device_activity -> devices: check that every activity device_id exists in devices.
SELECT a.*
FROM device_activity AS a
WHERE NOT EXISTS (
    SELECT 1
    FROM devices AS d
    WHERE d.device_id = a.device_id
)
ORDER BY a.device_id, a.activity_date;


device_id,activity_date,sessions,minutes_watched


In [7]:
%%sql
-- devices -> partners: check that every non-NULL partner_id exists in partners.
SELECT d.*
FROM devices AS d
WHERE d.partner_id IS NOT NULL
  AND NOT EXISTS (
      SELECT 1
      FROM partners AS p
      WHERE p.partner_id = d.partner_id
  )
ORDER BY d.device_id;


device_id,brand,country,partner_id,activation_date


In [8]:
%%sql
-- devices: count records with no assigned partner_id.
SELECT COUNT(*) AS devices_without_partner
FROM devices
WHERE partner_id IS NULL;


devices_without_partner
1


In [9]:
%%sql
-- device_activity + devices + revenue_by_market: compare activity and revenue coverage by country and month for July-September 2026.
WITH activity_months AS (
    SELECT DISTINCT
        CAST(DATE_TRUNC('month', a.activity_date) AS DATE) AS month,
        d.country
    FROM device_activity AS a
    JOIN devices AS d USING (device_id)
),
revenue_months AS (
    SELECT DISTINCT
        CAST(DATE_TRUNC('month', revenue_month) AS DATE) AS month,
        country
    FROM revenue_by_market
)
SELECT
    month,
    country,
    a.month IS NOT NULL AS has_activity,
    r.month IS NOT NULL AS has_revenue
FROM activity_months AS a
FULL OUTER JOIN revenue_months AS r USING (month, country)
WHERE month >= DATE '2026-07-01'
  AND month < DATE '2026-10-01'
  AND (a.month IS NULL OR r.month IS NULL)
ORDER BY month, country;


month,country,has_activity,has_revenue


In [10]:
%%sql
-- partners: check for missing partner_name and revenue_share_pct.
SELECT *
FROM partners
WHERE NULLIF(TRIM(partner_name), '') IS NULL
   OR revenue_share_pct IS NULL
ORDER BY partner_id;


partner_id,partner_name,revenue_share_pct


In [11]:
%%sql
-- devices: check for missing device_id, brand, country, and activation_date.
SELECT *
FROM devices
WHERE NULLIF(TRIM(device_id), '') IS NULL
   OR NULLIF(TRIM(brand), '') IS NULL
   OR NULLIF(TRIM(country), '') IS NULL
   OR activation_date IS NULL
ORDER BY device_id;


device_id,brand,country,partner_id,activation_date


In [12]:
%%sql
-- device_activity: check for missing device_id, activity_date, sessions, and minutes_watched.
SELECT *
FROM device_activity
WHERE NULLIF(TRIM(device_id), '') IS NULL
   OR activity_date IS NULL
   OR sessions IS NULL
   OR minutes_watched IS NULL
ORDER BY device_id, activity_date;


device_id,activity_date,sessions,minutes_watched


In [13]:
%%sql
-- revenue_by_market: check for missing revenue_month, country, product, and revenue_eur.
SELECT *
FROM revenue_by_market
WHERE revenue_month IS NULL
   OR NULLIF(TRIM(country), '') IS NULL
   OR NULLIF(TRIM(product), '') IS NULL
   OR revenue_eur IS NULL
ORDER BY revenue_month, country, product;


revenue_month,country,product,revenue_eur


In [14]:
%%sql
-- device_activity + devices: find activity_date earlier than the device activation_date.
SELECT
    a.*,
    d.activation_date
FROM device_activity AS a
JOIN devices AS d ON d.device_id = a.device_id
WHERE a.activity_date < d.activation_date
ORDER BY a.device_id, a.activity_date;


device_id,activity_date,sessions,minutes_watched,activation_date
D006,2026-05-27,1,30,2026-06-05


In [15]:
%%sql
-- revenue_by_market: check that revenue_month is the first day of the month.
SELECT *
FROM revenue_by_market
WHERE EXTRACT(DAY FROM revenue_month) <> 1
ORDER BY revenue_month, country, product;


revenue_month,country,product,revenue_eur


In [16]:
%%sql
-- partners: find revenue_share_pct outside the inclusive range from 0 to 1.
SELECT *
FROM partners
WHERE revenue_share_pct < 0
   OR revenue_share_pct > 1
ORDER BY partner_id;


partner_id,partner_name,revenue_share_pct


In [17]:
%%sql
-- device_activity: find negative sessions or minutes_watched.
SELECT *
FROM device_activity
WHERE sessions < 0
   OR minutes_watched < 0
ORDER BY device_id, activity_date;


device_id,activity_date,sessions,minutes_watched


In [18]:
%%sql
-- device_activity: find positive minutes_watched with zero sessions.
SELECT *
FROM device_activity
WHERE sessions = 0
  AND minutes_watched > 0
ORDER BY device_id, activity_date;


device_id,activity_date,sessions,minutes_watched


In [19]:
%%sql
-- revenue_by_market: find negative revenue_eur.
SELECT *
FROM revenue_by_market
WHERE revenue_eur < 0
ORDER BY revenue_month, country, product;


revenue_month,country,product,revenue_eur


In [20]:
%%sql
-- devices: check allowed country and brand values, including casing and extra spaces.
SELECT *
FROM devices
WHERE country NOT IN ('ES', 'FR', 'DE')
   OR brand NOT IN ('Brand A', 'Brand B', 'Brand C')
ORDER BY device_id;


device_id,brand,country,partner_id,activation_date


In [21]:
%%sql
-- revenue_by_market: check allowed country and product values, including casing and extra spaces.
SELECT *
FROM revenue_by_market
WHERE country NOT IN ('ES', 'FR', 'DE')
   OR product NOT IN ('FAST', 'Display Ads', 'Apps')
ORDER BY revenue_month, country, product;


revenue_month,country,product,revenue_eur
